# Notebook 03 — Annotation Processing (LIDC-IDRI)
**Dissertation:** Explainable Deep Transfer Learning Framework for Lung CT Image Classification
**Student:** Ranjeet Kumar Mahato (u3035175)

Follows on from your Notebook 01/02 (DICOM loading → HU conversion → lung windowing → resampling).
This notebook will:

1. read your 10-patient CT subset,

2. find and parse the XML annotation files,

3. match XML annotations to the correct CT series,

4. extract radiologist nodule information,

5. compute malignancy-based labels,

6. save nodule crops,

7. resize them for CNN training,

8. and create a patient-level train/val/test split.




## 1. Setup

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
!pip -q install pydicom pylidc scikit-learn opencv-python-headless matplotlib seaborn pandas numpy tqdm
print("Packages ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 77.0 MB/s eta 0:00:00
Packages ready.


###Import Required Libraries

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
from tqdm import tqdm
import os

##Define Dataset Paths

In [5]:
DATASET_PATH = Path("/content/drive/MyDrive/Dissertation/Dataset")

RAW_PATH = DATASET_PATH / "Raw"

XML_PATH = DATASET_PATH / "ANNOTATION_STAGING"

PROCESSED_PATH = DATASET_PATH / "Processed"

print("Raw:", RAW_PATH.exists())
print("XML:", XML_PATH.exists())
print("Processed:", PROCESSED_PATH.exists())

Raw: True
XML: True
Processed: True


## 3. Locate Every XML File

In [6]:
xml_files = sorted(XML_PATH.rglob("*.xml"))

print("Total XML files:", len(xml_files))

print("\nFirst five XML files:")

for file in xml_files[:5]:
    print(file)

Total XML files: 1319

First five XML files:
/content/drive/MyDrive/Dissertation/Dataset/ANNOTATION_STAGING/LIDC-XML-only/161-resubmitted-correction-3-9-12.xml
/content/drive/MyDrive/Dissertation/Dataset/ANNOTATION_STAGING/LIDC-XML-only/tcia-lidc-xml/157/158.xml
/content/drive/MyDrive/Dissertation/Dataset/ANNOTATION_STAGING/LIDC-XML-only/tcia-lidc-xml/157/159.xml
/content/drive/MyDrive/Dissertation/Dataset/ANNOTATION_STAGING/LIDC-XML-only/tcia-lidc-xml/157/160.xml
/content/drive/MyDrive/Dissertation/Dataset/ANNOTATION_STAGING/LIDC-XML-only/tcia-lidc-xml/157/161.xml


## 4. Read One XML File

In [7]:
sample_xml = xml_files[0]

tree = ET.parse(sample_xml)

root = tree.getroot()

print(root.tag)

print("\nChildren:")

for child in root[:10]:
    print(child.tag)

{http://www.nih.gov}LidcReadMessage

Children:
{http://www.nih.gov}ResponseHeader
{http://www.nih.gov}readingSession
{http://www.nih.gov}readingSession
{http://www.nih.gov}readingSession
{http://www.nih.gov}readingSession


## 5. Display XML Hierarchy

In [9]:
def print_tree(element, level=0, max_depth=3):

    print("   " * level + element.tag)

    if level < max_depth:
        for child in element:
            print_tree(child, level + 1, max_depth)

print_tree(root)

{http://www.nih.gov}LidcReadMessage
   {http://www.nih.gov}ResponseHeader
      {http://www.nih.gov}Version
      {http://www.nih.gov}MessageId
      {http://www.nih.gov}DateRequest
      {http://www.nih.gov}TimeRequest
      {http://www.nih.gov}RequestingSite
      {http://www.nih.gov}ServicingSite
      {http://www.nih.gov}TaskDescription
      {http://www.nih.gov}CtImageFile
      {http://www.nih.gov}SeriesInstanceUid
      {http://www.nih.gov}StudyInstanceUID
      {http://www.nih.gov}DateService
      {http://www.nih.gov}TimeService
      {http://www.nih.gov}ResponseDescription
      {http://www.nih.gov}ResponseComments
   {http://www.nih.gov}readingSession
      {http://www.nih.gov}annotationVersion
      {http://www.nih.gov}servicingRadiologistID
      {http://www.nih.gov}unblindedReadNodule
         {http://www.nih.gov}noduleID
         {http://www.nih.gov}roi
      {http://www.nih.gov}unblindedReadNodule
         {http://www.nih.gov}noduleID
         {http://www.nih.gov}roi
  

##6. Define the XML Namespace

In [15]:
# XML namespace used by LIDC-IDRI annotations
ns = {
    "nih": "http://www.nih.gov"
}

## 7. Extract Patient Information

In [11]:

study_uid = root.find(".//nih:StudyInstanceUID", ns)

series_uid = root.find(".//nih:SeriesInstanceUid", ns)

print("Study UID :", study_uid.text if study_uid is not None else "Not Found")

print("Series UID:", series_uid.text if series_uid is not None else "Not Found")

Study UID : 1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051825667176600857752
Series UID: 1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094259402036602717327


## 8. Count Reading Sessions

In [14]:
reading_sessions = root.findall(".//nih:readingSession", ns)

print("Number of Reading Sessions:", len(reading_sessions))

Number of Reading Sessions: 4


## 9. Count Annotated Nodules

In [16]:
nodule_count = 0

for session in reading_sessions:

    nodules = session.findall(".//nih:unblindedReadNodule", ns)

    nodule_count += len(nodules)

print("Total Nodule Annotations:", nodule_count)

Total Nodule Annotations: 21


## 10. Inspect the First Nodule


In [17]:
first_session = reading_sessions[0]

first_nodule = first_session.find(".//nih:unblindedReadNodule", ns)

for child in first_nodule:
    print(child.tag)

{http://www.nih.gov}noduleID
{http://www.nih.gov}roi


##11. Inspect the First Nodule Completely

In [22]:
first_session = reading_sessions[0]

first_nodule = first_session.find("nih:unblindedReadNodule", ns)

for child in first_nodule:
    print("=" * 50)
    print("Tag :", child.tag)
    print("Text:", child.text)

Tag : {http://www.nih.gov}noduleID
Text: 0
Tag : {http://www.nih.gov}roi
Text: 
    


##12. Extract All Characteristics of One Nodule

In [23]:
characteristics = first_nodule.find("nih:characteristics", ns)

if characteristics is None:
    print("No characteristics found.")
else:
    for feature in characteristics:
        print(f"{feature.tag.split('}')[-1]:20s} {feature.text}")

No characteristics found.
